In [8]:
import pandas as pd
import json
from pathlib import Path

meta_path = Path("../data/sampled/retail_sample_10_meta.csv")
flow_path = Path("../data/sampled/retail_sample_10_flow.csv")
save_dir = Path("../data/sampled")
save_dir.mkdir(parents=True, exist_ok=True)

df_meta = pd.read_csv(meta_path)
df_flow = pd.read_csv(flow_path)
df_flow["flow_id"] = df_flow.index.astype(str)
df_flow = df_flow.rename(columns={"fkey": "meta_id"})
df_meta = df_meta.rename(columns={"fkey": "meta_id"})

dataset_meta = {
    "relation_order": [[None, "meta"], ["meta", "flow"]],
    "tables": {
        "meta": {"children": ["flow"], "parents": []},
        "flow": {"children": [], "parents": ["meta"]}
    }
}
with open(save_dir / "dataset_meta.json", "w") as f:
    json.dump(dataset_meta, f, indent=4)

def get_domain(df):
    domain = {}
    for col in df.columns:
        if col.endswith("_id"):
            col_type = "discrete"
            size = df[col].nunique()
        else:
            dtype = df[col].dtype
            if pd.api.types.is_numeric_dtype(dtype):
                col_type = "continuous"
                size = min(100, df[col].nunique())
            else:
                col_type = "discrete"
                size = df[col].nunique()
        domain[col] = {"size": int(size), "type": col_type}
    return domain

meta_domain = get_domain(df_meta)
with open(save_dir / "meta_domain.json", "w") as f:
    json.dump(meta_domain, f, indent=4)

flow_domain = get_domain(df_flow)
with open(save_dir / "flow_domain.json", "w") as f:
    json.dump(flow_domain, f, indent=4)


In [ ]:
from clava_main import clava_clustering, clava_training, clava_synthesizing, clava_eval, load_configs

clustering_start_time = time.time()
configs, save_dir = load_configs("../data/sampled/retail_clava.json")
tables, relation_order, dataset_meta = load_multi_table(configs['general']['data_dir'])

tables, all_group_lengths_prob_dicts = clava_clustering(tables, relation_order, save_dir, configs)
clustering_end_time = time.time()
clustering_time_spent = clustering_end_time - clustering_start_time
training_start_time = time.time()

tables, models = clava_training(tables, relation_order, save_dir, configs)
training_end_time = time.time()
training_time_spent = training_end_time - training_start_time

cleaned_tables, synthesizing_time_spent, matching_time_spent = clava_synthesizing(
    tables, 
    relation_order, 
    save_dir, 
    all_group_lengths_prob_dicts, 
    models,
    configs,
    sample_scale=1 if not 'debug' in configs else configs['debug']['sample_scale']
)

report = clava_eval(tables, save_dir, configs, relation_order, cleaned_tables)
print('Time spent: ')
print('Clustering: ', clustering_time_spent)
print('Training: ', training_time_spent)
print('Synthesizing: ', synthesizing_time_spent)
print('Matching: ', matching_time_spent)
print('Total: ', clustering_time_spent + training_time_spent + synthesizing_time_spent + matching_time_spent)
